In [2]:
import pickle
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import root_mean_squared_error

from var import DATA_OUT, START_DATE

In [3]:
MAV_WINDOW = 5
LAG_WINDOW = 180
HORIZON = 10 # Numero di step futuri da predire

max_window = max(MAV_WINDOW, LAG_WINDOW)

df = pd.read_parquet(Path(DATA_OUT, 'df.parquet'), engine='pyarrow').drop(
    columns=['perc_mild_scint', 'perc_strong_scint', 's4_max']
)

# Pre-filtering
df = df[(df.index.hour > 17) | (df.index.hour < 6)]
# MAVs
df[f"s4_mean_ema_{MAV_WINDOW}m"] = (
    df['s4_mean'].ewm(span=MAV_WINDOW).mean()
)
# Lags
df[f"h_tmk_lag_{LAG_WINDOW}m"] = (
    df['h_tmk'].shift(LAG_WINDOW)
)
# Final filtering
df = df[(df.index.hour >= 18) | (df.index.hour < 6)]

# Multi-step target
y_cols = [f's4_mean_{i+1}' for i in range(HORIZON)]
for i in range(HORIZON):
    df[y_cols[i]] = df['s4_mean'].shift(-(i+1))

In [13]:
TRAIN_START, TRAIN_STOP = '2022-10-01', '2023-03-17'
TEST_START, TEST_STOP = '2023-03-18', '2023-03-31'

X_cols = [
    'n_sat',
    's4_mean',
    'field_magnitude_avg',
    'wind_speed',
    'wind_density',
    'wind_pressure',
    'eletric_field',
    'h_tmk',
    'f10.7_adj',
    'sza',
    's4_mean_ema_5m',
    'h_tmk_lag_180m',
]

X_train, X_test = df.loc[TRAIN_START:TRAIN_STOP, X_cols].copy(), df.loc[TEST_START:TEST_STOP, X_cols].copy()
y_train, y_test = df.loc[TRAIN_START:TRAIN_STOP, y_cols].copy().fillna(0), df.loc[TEST_START:TEST_STOP, y_cols].copy().fillna(0)

In [5]:
# n_splits = 4
# tscv = TimeSeriesSplit(n_splits=n_splits)

# rf_model = RandomForestRegressor(random_state=42)
# rf_params = {
#     "max_depth": [int(x) for x in np.linspace(2, 8, num=4)],
#     "n_estimators": [int(x) for x in np.linspace(20, 300, num=15)],
#     "min_samples_split": [int(x) for x in np.linspace(2, 80, num=3)]
# }

# cv_obj = GridSearchCV(
#     rf_model,
#     param_grid=rf_params,
#     cv=tscv,
#     scoring="neg_root_mean_squared_error",
#     verbose=2,
#     n_jobs=-1,
# )

# cv_obj.fit(X_train, y_train)
# rf_best_params = cv_obj.best_params_

In [6]:
# rf_best_params
# {'max_depth': 6, 'min_samples_split': 80, 'n_estimators': 300}

In [14]:
rf = RandomForestRegressor(
    **{'max_depth': 6, 'min_samples_split': 80, 'n_estimators': 300},
    random_state=42
)
rf.fit(X_train, y_train)

RandomForestRegressor(max_depth=6, min_samples_split=80, n_estimators=300,
                      random_state=42)

In [15]:
y_pred = rf.predict(X_test)

# with open(Path(DATA_OUT, 'rf_mimo_model.pkl'), 'wb') as f:
#     pickle.dump(rf, f)

In [16]:
df_eval = pd.concat(
    [
        y_test,
        pd.DataFrame(
            y_pred, columns=[f's4_mean_pred_{i+1}' for i in range(HORIZON)], index=y_test.index
        ),
    ],
    axis=1,
)

In [17]:
rmse_vals = []
rrmse_vals = []
steps = y_test.shape[1]

for i in range(steps):
    rmse = root_mean_squared_error(df_eval.iloc[:, i], df_eval.iloc[:, i+steps])
    rrmse = rmse / y_train.iloc[:, i].mean()
    
    rmse_vals.append(rmse)
    rrmse_vals.append(rrmse)

In [ ]:
rmse_vals

[0.03493836296457978,
 0.047507673143890004,
 0.05593175338363922,
 0.061884919003747145,
 0.066833108574429,
 0.07016100044354985,
 0.07260064774378053,
 0.0745506820557977,
 0.07629789523614039,
 0.07801770019394645]

In [19]:
rrmse_vals

[0.5348091418446004,
 0.7300519134035802,
 0.8598076978647409,
 0.9517250606119769,
 1.027366239071443,
 1.0781286405025299,
 1.1143444990053175,
 1.142191974927013,
 1.1700138942891063,
 1.1963147836026513]

In [ ]:
px.scatter(df_eval, x='s4_mean_1', y='s4_mean_pred_1')

In [ ]:
px.scatter(df_eval, x='s4_mean_3', y='s4_mean_pred_3')